In [1]:
# Imports

import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
import time
import collections
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow  : {tf.__version__}")
print(f"OpenCV      : {cv2.__version__}")
print(f"MediaPipe   : {mp.__version__}")

2026-10-12 17:15:18.278690: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-12 17:15:18.657474: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-12 17:15:18.659141: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-12 17:15:19.385169: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-12 17:15:23.848999: W tensorflow/compiler/tf

TensorFlow  : 2.16.2
OpenCV      : 4.11.0
MediaPipe   : 0.10.21


In [2]:
# Intial config 

# ── Paths ──────────────────────────────────────────────────────────────────
MODELS_DIR  = "/home/user/22059240/FYP/models"
BEST_MODEL_NAME_FILE = MODELS_DIR + "/best_model_name.txt"

# ── Feature pipeline (must mirror data_preprocessing.ipynb exactly) ────────
NUM_KEYPOINTS  = 33
COORDS_PER_KP  = 4          # x, y, z, visibility
WINDOW_SIZE    = 30          # frames per inference window
FEATURE_DIM    = 166         # 132 kp + 33 vel + 1 aspect ratio
VISIBILITY_THR = 0.5         # same as preprocessing

# ── Inference controls ─────────────────────────────────────────────────────
FALL_THRESHOLD       = 0.5   # sigmoid probability above which a window = FALL
CONFIRM_CONSECUTIVE  = 3     # number of consecutive FALL windows before alert
DISPLAY_WIDTH        = 960
DISPLAY_HEIGHT       = 540

# ── Colours (BGR) ──────────────────────────────────────────────────────────
COLOR_NORMAL = (0, 200, 0)    # green  – no fall
COLOR_ALERT  = (0, 0, 220)    # red    – fall confirmed
COLOR_WARN   = (0, 165, 255)  # orange – fall detected, not yet confirmed

In [4]:
# Test - can delete later

import os

print("MODELS_DIR:", MODELS_DIR)
print("\nFiles in models folder:")

if os.path.exists(MODELS_DIR):
    for f in os.listdir(MODELS_DIR):
        print(f)
else:
    print("Models folder does not exist!")

MODELS_DIR: /home/user/22059240/FYP/models

Files in models folder:
cnn_lstm_best.keras
simple_lstm_final.keras
gru_final.keras
stacked_lstm_best.keras
cnn_lstm_final.keras
bilstm_best.keras
gru_best.keras
best_model_final.keras
stacked_lstm_final.keras
bilstm_final.keras
simple_lstm_best.keras
best_model_name.txt


In [5]:
#Test 2 - can delete later 

print("BEST_MODEL_NAME_FILE:", BEST_MODEL_NAME_FILE)
print("File exists:", os.path.exists(BEST_MODEL_NAME_FILE))

if os.path.exists(BEST_MODEL_NAME_FILE):
    with open(BEST_MODEL_NAME_FILE, "r") as f:
        print("Selected model name:", repr(f.read().strip()))

BEST_MODEL_NAME_FILE: /home/user/22059240/FYP/models/best_model_name.txt
File exists: True
Selected model name: 'BiLSTM'


In [6]:
# Load best model + initialise MediaPipe

# Load best model (saved by model_evaluation.ipynb)
with open(BEST_MODEL_NAME_FILE, "r") as f:
    best_model_name = f.read().strip()

model_path = f"{MODELS_DIR}/{best_model_name.lower()}_final.keras"
model = tf.keras.models.load_model(model_path)
print(f"Loaded model : {best_model_name}")
print(f"Input shape  : {model.input_shape}")

# MediaPipe BlazePose — identical settings to data_preprocessing.ipynb
mp_pose    = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_styles  = mp.solutions.drawing_styles

pose_detector = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    enable_segmentation=False,
    min_detection_confidence=0.3,
    min_tracking_confidence=0.3
)
print("BlazePose initialised.")

2026-10-12 17:18:48.399220: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Loaded model : BiLSTM
Input shape  : (None, 30, 166)
BlazePose initialised.


I0000 00:00:1791796729.516058   31981 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1791796729.644438   37207 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 570.172.08), renderer: NVIDIA L4/PCIe/SSE2


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1791796729.801407   37202 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791796729.837322   37201 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [7]:
#Feature extraction helpers

def extract_keypoints(frame_rgb):
    """Run BlazePose on one RGB frame. Returns (kp_vector [132,], detected bool)."""
    results = pose_detector.process(frame_rgb)
    if results.pose_landmarks is None:
        return np.zeros(NUM_KEYPOINTS * COORDS_PER_KP, dtype=np.float32), False, None
    kp_vector = []
    for lm in results.pose_landmarks.landmark:
        kp_vector.extend([lm.x, lm.y, lm.z, lm.visibility])
    return np.array(kp_vector, dtype=np.float32), True, results.pose_landmarks


def normalize_keypoints(kp_vector):
    """Person-relative bounding box normalisation — identical to preprocessing."""
    kp = kp_vector.reshape(NUM_KEYPOINTS, COORDS_PER_KP).copy()
    visible_mask = kp[:, 3] >= VISIBILITY_THR
    if visible_mask.sum() < 2:
        return kp.flatten()
    xs = kp[visible_mask, 0]
    ys = kp[visible_mask, 1]
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    scale = max(x_max - x_min, y_max - y_min, 1e-6)
    kp[:, 0] = (kp[:, 0] - x_min) / scale
    kp[:, 1] = (kp[:, 1] - y_min) / scale
    return kp.flatten()


def compute_aspect_ratio(kp_vector):
    """Body bounding box aspect ratio (height / width) — identical to preprocessing."""
    kp = kp_vector.reshape(NUM_KEYPOINTS, COORDS_PER_KP)
    visible_mask = kp[:, 3] >= VISIBILITY_THR
    if visible_mask.sum() < 2:
        return 1.0
    xs = kp[visible_mask, 0]
    ys = kp[visible_mask, 1]
    width  = xs.max() - xs.min()
    height = ys.max() - ys.min()
    if width < 1e-6:
        return 1.0
    return float(min(height / width, 5.0))


def build_feature_vector(kp_norm, prev_kp_norm):
    """Assemble the 166-dim vector for one frame given current and previous keypoints."""
    y_indices = [i * COORDS_PER_KP + 1 for i in range(NUM_KEYPOINTS)]
    if prev_kp_norm is None:
        vel = np.zeros(NUM_KEYPOINTS, dtype=np.float32)
    else:
        vel = (kp_norm[y_indices] - prev_kp_norm[y_indices]).astype(np.float32)
    ar = compute_aspect_ratio(kp_norm)
    return np.concatenate([kp_norm, vel, [ar]]).astype(np.float32)

In [8]:
# Overlay

def draw_overlay(frame, status, prob, fall_count, fps):
    """
    Draw status banner, probability bar, fall counter and FPS onto the frame.
    status : "NORMAL" | "WARNING" | "FALL DETECTED"
    """
    h, w = frame.shape[:2]

    # ── Status banner ──────────────────────────────────────────────────────
    if status == "FALL DETECTED":
        color = COLOR_ALERT
    elif status == "WARNING":
        color = COLOR_WARN
    else:
        color = COLOR_NORMAL

    cv2.rectangle(frame, (0, 0), (w, 55), color, -1)
    cv2.putText(frame, status, (12, 38),
                cv2.FONT_HERSHEY_DUPLEX, 1.1, (255, 255, 255), 2, cv2.LINE_AA)

    # ── Probability bar ────────────────────────────────────────────────────
    bar_x, bar_y, bar_w, bar_h = 12, 68, 260, 18
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (60, 60, 60), -1)
    fill = int(bar_w * prob)
    bar_color = COLOR_ALERT if prob >= FALL_THRESHOLD else COLOR_NORMAL
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + fill, bar_y + bar_h), bar_color, -1)
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (200, 200, 200), 1)
    cv2.putText(frame, f"Fall prob: {prob:.2f}", (bar_x, bar_y + bar_h + 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (230, 230, 230), 1, cv2.LINE_AA)

    # ── Fall event counter ─────────────────────────────────────────────────
    cv2.putText(frame, f"Fall events: {fall_count}", (12, bar_y + bar_h + 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (230, 230, 230), 1, cv2.LINE_AA)

    # ── FPS ────────────────────────────────────────────────────────────────
    cv2.putText(frame, f"FPS: {fps:.1f}", (w - 110, h - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1, cv2.LINE_AA)

    return frame

In [ ]:
# Real time detection loop

# ── Choose source: 0 = webcam, or pass a video file path ──────────────────
VIDEO_SOURCE = 0   # change to "/home/user/22059240/FYP/data/URFD/fall/seq_01" etc.
                   # for testing with a saved video file instead of webcam

cap = cv2.VideoCapture(VIDEO_SOURCE)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video source: {VIDEO_SOURCE}")

# ── Rolling buffer: keeps last WINDOW_SIZE feature vectors ────────────────
frame_buffer  = collections.deque(maxlen=WINDOW_SIZE)
prev_kp_norm  = None    # for velocity computation

# ── State tracking ────────────────────────────────────────────────────────
consecutive_falls = 0   # consecutive windows predicted as fall
fall_event_count  = 0   # total confirmed fall events
current_prob      = 0.0
status            = "NORMAL"
in_fall_event     = False

# ── FPS counter ───────────────────────────────────────────────────────────
fps_timer   = time.time()
fps_counter = 0
fps_display = 0.0

print("Starting real-time detection. Press 'q' to quit.")
print(f"Model: {best_model_name}  |  Window: {WINDOW_SIZE} frames  |  Confirm threshold: {CONFIRM_CONSECUTIVE} windows")

while True:
    ret, frame = cap.read()
    if not ret:
        print("End of video / camera disconnected.")
        break

    frame = cv2.resize(frame, (DISPLAY_WIDTH, DISPLAY_HEIGHT))
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # ── Pose estimation ───────────────────────────────────────────────────
    kp_raw, detected, landmarks = extract_keypoints(frame_rgb)

    if detected:
        kp_norm = normalize_keypoints(kp_raw)
        fv = build_feature_vector(kp_norm, prev_kp_norm)
        prev_kp_norm = kp_norm
    else:
        fv = np.zeros(FEATURE_DIM, dtype=np.float32)
        prev_kp_norm = None

    frame_buffer.append(fv)

    # ── Draw skeleton when pose is detected ───────────────────────────────
    if detected and landmarks is not None:
        mp_drawing.draw_landmarks(
            frame,
            landmarks,
            mp_pose.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing.DrawingSpec(
                color=(255, 220, 100), thickness=2, circle_radius=3),
            connection_drawing_spec=mp_drawing.DrawingSpec(
                color=(180, 180, 180), thickness=1)
        )

    # ── Run inference only when buffer is full (30 frames) ────────────────
    INFER_EVERY_N = 5   # only run model every 5 frames

    if len(frame_buffer) == WINDOW_SIZE and fps_counter % INFER_EVERY_N == 0:
        window = np.array(frame_buffer, dtype=np.float32)   # (30, 166)
        window_input = window[np.newaxis, ...]               # (1, 30, 166)

        t0 = time.time()
        pred_prob = float(model(window_input, training=False).numpy()[0][0])
        latency_ms = (time.time() - t0) * 1000

        current_prob = pred_prob
        is_fall = pred_prob >= FALL_THRESHOLD

        if is_fall:
            consecutive_falls += 1
            if consecutive_falls >= CONFIRM_CONSECUTIVE:
                status = "FALL DETECTED"
                if not in_fall_event:
                    fall_event_count += 1
                    in_fall_event = True
            else:
                status = "WARNING"
        else:
            consecutive_falls = 0
            status = "NORMAL"
            in_fall_event = False

    # ── FPS ───────────────────────────────────────────────────────────────
    fps_counter += 1
    elapsed = time.time() - fps_timer
    if elapsed >= 1.0:
        fps_display = fps_counter / elapsed
        fps_counter = 0
        fps_timer   = time.time()

    # ── Draw overlay ──────────────────────────────────────────────────────
    frame = draw_overlay(frame, status, current_prob, fall_event_count, fps_display)

    cv2.imshow("Fall Detection — Real Time", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        print("Quit by user.")
        break
    elif key == ord('r'):
        # Press 'r' to reset all counters mid-session
        frame_buffer.clear()
        consecutive_falls = 0
        fall_event_count  = 0
        in_fall_event     = False
        current_prob      = 0.0
        status            = "NORMAL"
        print("Counters reset.")

cap.release()
cv2.destroyAllWindows()
pose_detector.close()
print(f"\nSession ended. Total fall events detected: {fall_event_count}")

[ WARN:0@233.037] global cap_v4l.cpp:913 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ERROR:0@233.043] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range


RuntimeError: Cannot open video source: 0